# 📈 Lesson 3.3 — Regression: Predicting Numbers

**AI Crash Course for Absolute Beginners**

When the answer is a **number** (not a category), we use regression. In this notebook:
- Understand the cost function and gradient descent visually
- Train Linear, Polynomial, and Ridge regression models
- Evaluate with MAE, MSE, RMSE, and R²
- Predict real house prices

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np                          # fast maths and arrays
import pandas as pd                         # working with tables
import matplotlib.pyplot as plt             # drawing charts
from sklearn.datasets import fetch_california_housing  # a real housing dataset built into scikit-learn
from sklearn.model_selection import train_test_split   # split data into train/test
from sklearn.preprocessing import StandardScaler, PolynomialFeatures  # scaling and polynomial features
from sklearn.linear_model import LinearRegression, Ridge  # our regression models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # evaluation metrics

---
## Part 1 — Simple Linear Regression (from scratch intuition)

In [ ]:
# Simulate: study hours vs exam score
np.random.seed(7)   # ensures the same random data every time

# np.linspace creates 50 evenly spaced numbers between 1 and 10
hours  = np.linspace(1, 10, 50)
# Score = 40 + 5.5 * hours + some random noise (to simulate real messy data)
score  = 40 + 5.5 * hours + np.random.normal(0, 5, 50)

# --- Manually calculate the best fit line using the least squares formula ---
x_mean, y_mean = hours.mean(), score.mean()

# Slope formula: sum of (x - x_mean)(y - y_mean) divided by sum of (x - x_mean)^2
slope     = np.sum((hours - x_mean) * (score - y_mean)) / np.sum((hours - x_mean)**2)

# Intercept: where the line crosses the y-axis
intercept = y_mean - slope * x_mean

print(f"Equation: score = {intercept:.1f} + {slope:.2f} * hours")

# Create line data for plotting
x_line = np.linspace(0, 11, 100)
y_line = intercept + slope * x_line

plt.figure(figsize=(7, 4))
plt.scatter(hours, score, alpha=0.7, label="Student data", color="#1a6bc8")
plt.plot(x_line, y_line, color="#c8401a", label=f"y = {intercept:.1f} + {slope:.2f}x")
plt.xlabel("Hours studied")
plt.ylabel("Exam score")
plt.title("Simple Linear Regression")
plt.legend()
plt.tight_layout()
plt.show()

---
## Part 2 — The Cost Function: What Does 'Learning' Minimise?

In [ ]:
# MSE as a function of slope (holding intercept fixed)
slopes = np.linspace(2, 9, 200)
mse_vals = [np.mean((score - (intercept + s * hours))**2) for s in slopes]

plt.figure(figsize=(7, 4))
plt.plot(slopes, mse_vals, color="#2a8c5a", linewidth=2)
plt.axvline(slope, color="#c8401a", linestyle="--", label=f"Optimal slope = {slope:.2f}")
plt.xlabel("Slope value")
plt.ylabel("Mean Squared Error")
plt.title("Cost Function: MSE vs Slope")
plt.legend()
plt.tight_layout()
plt.show()
print("Gradient descent moves along this curve toward the minimum.")

---
## Part 3 — California Housing Dataset

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name="MedianHouseValue")   # in $100k

print(f"Shape: {X.shape}")
print("\nFeature descriptions:")
for fname, fdesc in zip(housing.feature_names, housing.DESCR.split('\n')[16:24]):
    print(f"  {fname:<12}: {fdesc.strip()}")
X.describe().round(2)

In [ ]:
# Prepare
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

---
## Model 1 — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

print("Feature coefficients:")
coef_df = pd.Series(lr.coef_, index=housing.feature_names).sort_values()
coef_df.plot(kind="barh", figsize=(7, 4), color="#1a6bc8")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression Coefficients")
plt.tight_layout()
plt.show()

In [ ]:
def eval_regression(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"  MAE  : ${mae*100_000:,.0f}  (average error in dollars)")
    print(f"  RMSE : ${rmse*100_000:,.0f}")
    print(f"  R²   : {r2:.3f}  (1.0 = perfect)")
    return {"Model": name, "MAE ($k)": f"{mae:.3f}", "RMSE ($k)": f"{rmse:.3f}", "R²": f"{r2:.3f}"}

results = [eval_regression(y_test, y_pred_lr, "Linear Regression")]

---
## Model 2 — Ridge Regression (with regularisation)

In [ ]:
# Try different alpha values to find the best regularisation strength
# alpha controls how strongly we penalise large weights — higher alpha = simpler model
alphas = [0.01, 0.1, 1, 10, 100]
ridge_results = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_s, y_train)
    pred  = ridge.predict(X_test_s)
    ridge_results.append((alpha, r2_score(y_test, pred)))

# max(..., key=lambda x: x[1]) finds the pair with the highest R² score
# lambda x: x[1] means "compare by the second element of each pair (the R² value)"
best_alpha, best_r2 = max(ridge_results, key=lambda x: x[1])
print(f"Best alpha = {best_alpha}  =>  R² = {best_r2:.3f}")

# Train final model with the best alpha
ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_train_s, y_train)
y_pred_ridge = ridge_best.predict(X_test_s)

results.append(eval_regression(y_test, y_pred_ridge, f"Ridge (alpha={best_alpha})"))

---
## Predicted vs Actual Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test, y_pred_lr, alpha=0.3, s=10, color="#1a6bc8", label="Linear")
ax.plot([0, 5], [0, 5], color="#c8401a", linestyle="--", label="Perfect prediction")
ax.set_xlabel("Actual House Value ($100k)")
ax.set_ylabel("Predicted Value ($100k)")
ax.set_title("Predicted vs Actual")
ax.legend()
plt.tight_layout()
plt.show()
print("Points on the red dashed line = perfect prediction.")
print("Spread away from it = error.")

In [ ]:
# Final comparison
pd.DataFrame(results).set_index("Model")

---
## Challenge Exercises

1. **Gradient Descent**: Implement mini gradient descent manually — update `slope = slope - lr * d_loss/d_slope` in a loop and watch MSE drop.
2. **Polynomial features**: `from sklearn.preprocessing import PolynomialFeatures` with `degree=2` — does it improve R²?
3. **Lasso**: Try `from sklearn.linear_model import Lasso` — some coefficients will be set to exactly 0 (automatic feature selection).

---
**Next lesson:** [Lesson 3.4 — Clustering](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-3.4-clustering.ipynb)